# Deep Q-Learning - Lunar Lander

In this assignment, you will train an agent to land a lunar lander safely on a landing pad on the surface of the moon.


# Outline
- [ 1 - Import Packages <img align="Right" src="./images/lunar_lander.gif" width = 60% >](#1)
- [ 2 - Hyperparameters](#2)
- [ 3 - The Lunar Lander Environment](#3)
  - [ 3.1 Action Space](#3.1)
  - [ 3.2 Observation Space](#3.2)
  - [ 3.3 Rewards](#3.3)
  - [ 3.4 Episode Termination](#3.4)
- [ 4 - Load the Environment](#4)
- [ 5 - Interacting with the Gym Environment](#5)
    - [ 5.1 Exploring the Environment's Dynamics](#5.1)
- [ 6 - Deep Q-Learning](#6)
  - [ 6.1 Target Network](#6.1)
    - [ Exercise 1](#ex01)
  - [ 6.2 Experience Replay](#6.2)
- [ 7 - Deep Q-Learning Algorithm with Experience Replay](#7)
  - [ Exercise 2](#ex02)
- [ 8 - Update the Network Weights](#8)
- [ 9 - Train the Agent](#9)
- [ 10 - See the Trained Agent In Action](#10)
- [ 11 - Congratulations!](#11)
- [ 12 - References](#12)


# 深度 Q 学习 - 月球着陆器（Lunar Lander）

在本作业中，你将训练一个智能体，使月球着陆器安全降落在月球表面的着陆坪上。


# 大纲
- [ 1 - 导入软件包](#1)
- [ 2 - 超参数](#2)
- [ 3 - 月球着陆器环境](#3)
  - [ 3.1 动作空间](#3.1)
  - [ 3.2 观测空间](#3.2)
  - [ 3.3 奖励](#3.3)
  - [ 3.4 回合终止](#3.4)
- [ 4 - 加载环境](#4)
- [ 5 - 与 Gym 环境交互](#5)
    - [ 5.1 探索环境的动态特性](#5.1)
- [ 6 - 深度 Q 学习](#6)
  - [ 6.1 目标网络](#6.1)
    - [ 练习 1](#ex01)
  - [ 6.2 经验回放](#6.2)
- [ 7 - 带经验回放的深度 Q 学习算法](#7)
  - [ 练习 2](#ex02)
- [ 8 - 更新网络权重](#8)
- [ 9 - 训练智能体](#9)
- [ 10 - 查看训练好的智能体的表现](#10)
- [ 11 - 恭喜！](#11)
- [ 12 - 参考文献](#12)

<a name="1"></a>
## 1 - Import Packages

We'll make use of the following packages:
- `numpy` is a package for scientific computing in python.
- `deque` will be our data structure for our memory buffer.
- `namedtuple` will be used to store the experience tuples.
- The `gym` toolkit is a collection of environments that can be used to test reinforcement learning algorithms. We should note that in this notebook we are using `gym` version `0.24.0`.
- `PIL.Image` and `pyvirtualdisplay` are needed to render the Lunar Lander environment.
- We will use several modules from the `tensorflow.keras` framework for building deep learning models.
- `utils` is a module that contains helper functions for this assignment. You do not need to modify the code in this file.

Run the cell below to import all the necessary packages.

## 1 - 导入软件包

我们将使用以下软件包：
- `numpy` 是 Python 中用于科学计算的软件包。
- `deque` 将作为我们记忆缓冲区（memory buffer）的数据结构。
- `namedtuple` 将用于存储经验元组（experience tuples）。
- `gym` 工具包是一组可用于测试强化学习算法的环境集合。需要注意，在本 notebook 中我们使用的是 `gym` 版本 `0.24.0`。
- `PIL.Image` 和 `pyvirtualdisplay` 用于渲染月球着陆器（Lunar Lander）环境。
- 我们将使用 `tensorflow.keras` 框架中的多个模块来构建深度学习模型。
- `utils` 是一个包含本次作业辅助函数的模块。你不需要修改该文件中的代码。

运行下方单元格以导入所有必要的软件包。

In [ ]:
import time
from collections import deque, namedtuple

import gym
import numpy as np
import PIL.Image
import tensorflow as tf
import utils

from pyvirtualdisplay import Display
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import MSE
from tensorflow.keras.optimizers import Adam

In [ ]:
# Set up a virtual display to render the Lunar Lander environment.
Display(visible=0, size=(840, 480)).start();

# Set the random seed for TensorFlow
tf.random.set_seed(utils.SEED)

<a name="2"></a>
## 2 - Hyperparameters

Run the cell below to set the hyperparameters.

## 2 - 超参数

运行下方单元格以设置超参数。

In [ ]:
MEMORY_SIZE = 100_000     # size of memory buffer
GAMMA = 0.995             # discount factor
ALPHA = 1e-3              # learning rate  
NUM_STEPS_FOR_UPDATE = 4  # perform a learning update every C time steps

<a name="3"></a>
## 3 - The Lunar Lander Environment

In this notebook we will be using [OpenAI's Gym Library](https://www.gymlibrary.ml/). The Gym library provides a wide variety of environments for reinforcement learning. To put it simply, an environment represents a problem or task to be solved. In this notebook, we will try to solve the Lunar Lander environment using reinforcement learning.

The goal of the Lunar Lander environment is to land the lunar lander safely on the landing pad on the surface of the moon. The landing pad is designated by two flag poles and it is always at coordinates `(0,0)` but the lander is also allowed to land outside of the landing pad. The lander starts at the top center of the environment with a random initial force applied to its center of mass and has infinite fuel. The environment is considered solved if you get `200` points. 

<br>
<br>
<figure>
  <img src = "images/lunar_lander.gif" width = 40%>
      <figcaption style = "text-align: center; font-style: italic">Fig 1. Lunar Lander Environment.</figcaption>
</figure>



<a name="3.1"></a>
### 3.1 Action Space

The agent has four discrete actions available:

* Do nothing.
* Fire right engine.
* Fire main engine.
* Fire left engine.

Each action has a corresponding numerical value:

```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```

<a name="3.2"></a>
### 3.2 Observation Space

The agent's observation space consists of a state vector with 8 variables:

* Its $(x,y)$ coordinates. The landing pad is always at coordinates $(0,0)$.
* Its linear velocities $(\dot x,\dot y)$.
* Its angle $\theta$.
* Its angular velocity $\dot \theta$.
* Two booleans, $l$ and $r$, that represent whether each leg is in contact with the ground or not.

<a name="3.3"></a>
### 3.3 Rewards

The Lunar Lander environment has the following reward system:

* Landing on the landing pad and coming to rest is about 100-140 points.
* If the lander moves away from the landing pad, it loses reward. 
* If the lander crashes, it receives -100 points.
* If the lander comes to rest, it receives +100 points.
* Each leg with ground contact is +10 points.
* Firing the main engine is -0.3 points each frame.
* Firing the side engine is -0.03 points each frame.

<a name="3.4"></a>
### 3.4 Episode Termination

An episode ends (i.e the environment enters a terminal state) if:

* The lunar lander crashes (i.e if the body of the lunar lander comes in contact with the surface of the moon).

* The lander's $x$-coordinate is greater than 1.

You can check out the [Open AI Gym documentation](https://www.gymlibrary.ml/environments/box2d/lunar_lander/) for a full description of the environment. 

## 3 - 月球着陆器（Lunar Lander）环境

在本 notebook 中，我们将使用 [OpenAI 的 Gym 库](https://www.gymlibrary.ml/)。Gym 库为强化学习提供了各种各样的环境。简单来说，环境代表一个待解决的问题或任务。在本 notebook 中，我们将尝试使用强化学习来解决月球着陆器环境。

月球着陆器环境的目标是将月球着陆器安全地降落在月球表面的着陆坪上。着陆坪由两根旗杆标出，始终位于坐标 `(0,0)` 处，但着陆器也允许降落在着陆坪之外。着陆器从环境顶部中央出发，其质心会受到一个随机的初始作用力，并且拥有无限燃料。如果你获得 `200` 分，则认为该环境已被解决。

### 3.1 动作空间

智能体有四个离散动作可供选择：

* 不执行任何操作（Do nothing）。
* 启动右侧引擎（Fire right engine）。
* 启动主引擎（Fire main engine）。
* 启动左侧引擎（Fire left engine）。

每个动作都有对应的数值：

```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```

### 3.2 观测空间

智能体的观测空间由包含 8 个变量的状态向量组成：

* 它的 $(x,y)$ 坐标。着陆坪始终位于坐标 $(0,0)$ 处。
* 它的线速度 $(\dot x,\dot y)$。
* 它的角度 $\theta$。
* 它的角速度 $\dot \theta$。
* 两个布尔值 $l$ 和 $r$，分别表示两条着陆腿是否与地面接触。

### 3.3 奖励

月球着陆器环境具有以下奖励机制：

* 降落在着陆坪并静止下来大约可获得 100-140 分。
* 如果着陆器远离着陆坪，则会失去奖励。
* 如果着陆器坠毁，则会获得 -100 分。
* 如果着陆器静止下来，则会获得 +100 分。
* 每条接触地面的着陆腿可获得 +10 分。
* 每帧启动主引擎会扣 0.3 分（-0.3 分）。
* 每帧启动侧引擎会扣 0.03 分（-0.03 分）。

### 3.4 回合终止

如果出现以下情况，一个回合（episode）将结束（即环境进入终止状态）：

* 月球着陆器坠毁（即月球着陆器的机身接触到月球表面）。

* 着陆器的 $x$ 坐标大于 1。

你可以查看 [OpenAI Gym 文档](https://www.gymlibrary.ml/environments/box2d/lunar_lander/) 以获取该环境的完整说明。

<a name="4"></a>
## 4 - Load the Environment

We start by loading the `LunarLander-v2` environment from the `gym` library by using the `.make()` method. `LunarLander-v2` is the latest version of the Lunar Lander environment and you can read about its version history in the [Open AI Gym documentation](https://www.gymlibrary.ml/environments/box2d/lunar_lander/#version-history).

## 4 - 加载环境

我们首先使用 `.make()` 方法从 `gym` 库中加载 `LunarLander-v2` 环境。`LunarLander-v2` 是月球着陆器环境的最新版本，你可以在 [OpenAI Gym 文档](https://www.gymlibrary.ml/environments/box2d/lunar_lander/#version-history) 中阅读其版本历史。

In [ ]:
env = gym.make('LunarLander-v2')

Once we load the environment we use the `.reset()` method to reset the environment to the initial state. The lander starts at the top center of the environment and we can render the first frame of the environment by using the `.render()` method.

加载环境后，我们使用 `.reset()` 方法将环境重置到初始状态。着陆器从环境顶部中央开始，我们可以使用 `.render()` 方法渲染环境的第一帧画面。

In [ ]:
env.reset()
PIL.Image.fromarray(env.render(mode='rgb_array'))

In order to build our neural network later on we need to know the size of the state vector and the number of valid actions. We can get this information from our environment by using the `.observation_space.shape` and `action_space.n` methods, respectively.

为了稍后构建神经网络，我们需要知道状态向量的大小以及有效动作的数量。我们可以分别使用 `.observation_space.shape` 和 `action_space.n` 从环境中获取这些信息。

In [ ]:
state_size = env.observation_space.shape
num_actions = env.action_space.n

print('State Shape:', state_size)
print('Number of actions:', num_actions)

<a name="5"></a>
## 5 - Interacting with the Gym Environment

The Gym library implements the standard “agent-environment loop” formalism:

<br>
<center>
<video src = "./videos/rl_formalism.m4v" width="840" height="480" controls autoplay loop poster="./images/rl_formalism.png"> </video>
<figcaption style = "text-align:center; font-style:italic">Fig 2. Agent-environment Loop Formalism.</figcaption>
</center>
<br>

In the standard “agent-environment loop” formalism, an agent interacts with the environment in discrete time steps $t=0,1,2,...$. At each time step $t$, the agent uses a policy $\pi$ to select an action $A_t$ based on its observation of the environment's state $S_t$. The agent receives a numerical reward $R_t$ and on the next time step, moves to a new state $S_{t+1}$.

<a name="5.1"></a>
### 5.1 Exploring the Environment's Dynamics

In Open AI's Gym environments, we use the `.step()` method to run a single time step of the environment's dynamics. In the version of `gym` that we are using the `.step()` method accepts an action and returns four values:

* `observation` (**object**): an environment-specific object representing your observation of the environment. In the Lunar Lander environment this corresponds to a numpy array containing the positions and velocities of the lander as described in section [3.2 Observation Space](#3.2).


* `reward` (**float**): amount of reward returned as a result of taking the given action. In the Lunar Lander environment this corresponds to a float of type `numpy.float64` as described in section [3.3 Rewards](#3.3).


* `done` (**boolean**): When done is `True`, it indicates the episode has terminated and it’s time to reset the environment. 


* `info` (**dictionary**): diagnostic information useful for debugging. We won't be using this variable in this notebook but it is shown here for completeness.

To begin an episode, we need to reset the environment to an initial state. We do this by using the `.reset()` method. 

## 5 - 与 Gym 环境交互

Gym 库实现了标准的“智能体-环境循环（agent-environment loop）”形式化框架：

在标准的“智能体-环境循环”形式化框架中，智能体在离散时间步 $t=0,1,2,...$ 上与环境交互。在每个时间步 $t$，智能体根据对环境状态 $S_t$ 的观测，使用策略 $\pi$ 选择一个动作 $A_t$。智能体会收到一个数值奖励 $R_t$，并在下一个时间步转移到新状态 $S_{t+1}$。

### 5.1 探索环境动态

在 OpenAI 的 Gym 环境中，我们使用 `.step()` 方法来运行环境动态的单个时间步。在我们使用的 `gym` 版本中，`.step()` 方法接收一个动作并返回四个值：

* `observation`（**object**）：特定于环境的对象，表示你对环境的观测。在月球着陆器环境中，它对应一个包含着陆器位置和速度的 NumPy 数组，详见 [3.2 观测空间](#3.2) 一节。


* `reward`（**float**）：执行给定动作后返回的奖励值。在月球着陆器环境中，它对应一个 `numpy.float64` 类型的浮点数，详见 [3.3 奖励](#3.3) 一节。


* `done`（**boolean**）：当 `done` 为 `True` 时，表示回合已终止，此时需要重置环境。


* `info`（**dictionary**）：有助于调试的诊断信息。我们在本 notebook 中不会使用该变量，这里列出仅为了保持完整性。

要开始一个回合，我们需要将环境重置为初始状态。这通过调用 `.reset()` 方法来实现。

In [ ]:
# Reset the environment and get the initial state.
initial_state = env.reset()

Once the environment is reset, the agent can start taking actions in the environment by using the `.step()` method. Note that the agent can only take one action per time step. 

In the cell below you can select different actions and see how the returned values change depending on the action taken. Remember that in this environment the agent has four discrete actions available and we specify them in code by using their corresponding numerical value:

```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```

环境重置后，智能体就可以使用 `.step()` 方法在环境中开始执行动作。请注意，智能体在每个时间步只能执行一个动作。

在下方单元格中，你可以选择不同的动作，观察返回值如何随所选动作而变化。请记住，在该环境中智能体有四个离散动作可用，我们在代码中通过它们对应的数值来指定：

```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```

In [ ]:
# Select an action
action = 0

# Run a single time step of the environment's dynamics with the given action.
next_state, reward, done, info = env.step(action)

with np.printoptions(formatter={'float': '{:.3f}'.format}):
    print("Initial State:", initial_state)
    print("Action:", action)
    print("Next State:", next_state)
    print("Reward Received:", reward)
    print("Episode Terminated:", done)
    print("Info:", info)

In practice, when we train the agent we use a loop to allow the agent to take many consecutive actions during an episode.

在实践中，训练智能体时我们会使用循环，让智能体在一个回合中连续执行许多动作。

<a name="6"></a>
## 6 - Deep Q-Learning

In cases where both the state and action space are discrete we can estimate the action-value function iteratively by using the Bellman equation:

$$
Q_{i+1}(s,a) = R + \gamma \max_{a'}Q_i(s',a')
$$

This iterative method converges to the optimal action-value function $Q^*(s,a)$ as $i\to\infty$. This means that the agent just needs to gradually explore the state-action space and keep updating the estimate of $Q(s,a)$ until it converges to the optimal action-value function $Q^*(s,a)$. However, in cases where the state space is continuous it becomes practically impossible to explore the entire state-action space. Consequently, this also makes it practically impossible to gradually estimate $Q(s,a)$ until it converges to $Q^*(s,a)$.

In the Deep $Q$-Learning, we solve this problem by using a neural network to estimate the action-value function $Q(s,a)\approx Q^*(s,a)$. We call this neural network a $Q$-Network and it can be trained by adjusting its weights at each iteration to minimize the mean-squared error in the Bellman equation.

Unfortunately, using neural networks in reinforcement learning to estimate action-value functions has proven to be highly unstable. Luckily, there's a couple of techniques that can be employed to avoid instabilities. These techniques consist of using a ***Target Network*** and ***Experience Replay***. We will explore these two techniques in the following sections.

## 6 - 深度 Q 学习（Deep Q-Learning）

当状态空间和动作空间都是离散的时候，我们可以利用贝尔曼方程（Bellman equation）迭代地估计动作价值函数：

$$
Q_{i+1}(s,a) = R + \gamma \max_{a'}Q_i(s',a')
$$

当 $i\to\infty$ 时，这种迭代方法会收敛到最优动作价值函数 $Q^*(s,a)$。这意味着智能体只需逐步探索状态-动作空间，并不断更新对 $Q(s,a)$ 的估计，直到收敛到最优动作价值函数 $Q^*(s,a)$。然而，在状态空间连续的情况下，几乎不可能探索整个状态-动作空间。因此，也就几乎不可能通过逐步估计 $Q(s,a)$ 来使其收敛到 $Q^*(s,a)$。

在深度 $Q$ 学习（Deep $Q$-Learning）中，我们通过使用神经网络来估计动作价值函数 $Q(s,a)\approx Q^*(s,a)$ 来解决这一问题。我们将这个神经网络称为 $Q$ 网络（$Q$-Network），它可以通过在每次迭代中调整权重以最小化贝尔曼方程中的均方误差来进行训练。

不幸的是，在强化学习中使用神经网络来估计动作价值函数已被证明是高度不稳定的。幸运的是，有几种技术可以用来避免不稳定性。这些技术包括使用***目标网络（Target Network）***和***经验回放（Experience Replay）***。我们将在接下来的小节中探讨这两种技术。

<a name="6.1"></a>
### 6.1 Target Network

We can train the $Q$-Network by adjusting it's weights at each iteration to minimize the mean-squared error in the Bellman equation, where the target values are given by:

$$
y = R + \gamma \max_{a'}Q(s',a';w)
$$

where $w$ are the weights of the $Q$-Network. This means that we are adjusting the weights $w$ at each iteration to minimize the following error:

$$
\overbrace{\underbrace{R + \gamma \max_{a'}Q(s',a'; w)}_{\rm {y~target}} - Q(s,a;w)}^{\rm {Error}}
$$

Notice that this forms a problem because the $y$ target is changing on every iteration. Having a constantly moving target can lead to oscillations and instabilities. To avoid this, we can create
a separate neural network for generating the $y$ targets. We call this separate neural network the **target $\hat Q$-Network** and it will have the same architecture as the original $Q$-Network. By using the target $\hat Q$-Network, the above error becomes:

$$
\overbrace{\underbrace{R + \gamma \max_{a'}\hat{Q}(s',a'; w^-)}_{\rm {y~target}} - Q(s,a;w)}^{\rm {Error}}
$$

where $w^-$ and $w$ are the weights the target $\hat Q$-Network and $Q$-Network, respectively.

In practice, we will use the following algorithm: every $C$ time steps we will use the $\hat Q$-Network to generate the $y$ targets and update the weights of the target $\hat Q$-Network using the weights of the $Q$-Network. We will update the weights $w^-$ of the the target $\hat Q$-Network using a **soft update**. This means that we will update the weights $w^-$ using the following rule:
 
$$
w^-\leftarrow \tau w + (1 - \tau) w^-
$$

where $\tau\ll 1$. By using the soft update, we are ensuring that the target values, $y$, change slowly, which greatly improves the stability of our learning algorithm.

### 6.1 目标网络（Target Network）

我们可以通过在每次迭代中调整 $Q$ 网络的权重来最小化贝尔曼方程中的均方误差，从而训练 $Q$ 网络，其中目标值由下式给出：

$$
y = R + \gamma \max_{a'}Q(s',a';w)
$$

其中 $w$ 是 $Q$ 网络的权重。这意味着我们在每次迭代中都在调整权重 $w$ 以最小化以下误差：

$$
\overbrace{\underbrace{R + \gamma \max_{a'}Q(s',a'; w)}_{\rm {y~target}} - Q(s,a;w)}^{\rm {Error}}
$$

注意这会带来一个问题，因为目标值 $y$ 在每次迭代中都在变化。目标不断移动会导致震荡和不稳定。为了避免这种情况，我们可以创建一个单独的神经网络来生成目标值 $y$。我们将这个单独的神经网络称为**目标 $\hat Q$ 网络（target $\hat Q$-Network）**，它具有与原始 $Q$ 网络相同的架构。使用目标 $\hat Q$ 网络后，上述误差变为：

$$
\overbrace{\underbrace{R + \gamma \max_{a'}\hat{Q}(s',a'; w^-)}_{\rm {y~target}} - Q(s,a;w)}^{\rm {Error}}
$$

其中 $w^-$ 和 $w$ 分别是目标 $\hat Q$ 网络和 $Q$ 网络的权重。

在实践中，我们将使用以下算法：每隔 $C$ 个时间步，我们使用 $\hat Q$ 网络生成目标值 $y$，并使用 $Q$ 网络的权重来更新目标 $\hat Q$ 网络的权重。我们将使用**软更新（soft update）**来更新目标 $\hat Q$ 网络的权重 $w^-$。这意味着我们将按照以下规则更新权重 $w^-$：

$$
w^-\leftarrow \tau w + (1 - \tau) w^-
$$

其中 $\tau\ll 1$。通过使用软更新，我们确保了目标值 $y$ 变化缓慢，从而大大提高了学习算法的稳定性。

<a name="ex01"></a>
### Exercise 1

In this exercise you will create the $Q$ and target $\hat Q$ networks and set the optimizer. Remember that the Deep $Q$-Network (DQN) is a neural network that approximates the action-value function $Q(s,a)\approx Q^*(s,a)$. It does this by learning how to map states to $Q$ values.

To solve the Lunar Lander environment, we are going to employ a DQN with the following architecture:

* An `Input` layer that takes `state_size` as input.

* A `Dense` layer with `64` units and a `relu` activation function.

* A `Dense` layer with `64` units and a `relu` activation function.

* A `Dense` layer with `num_actions` units and a `linear` activation function. This will be the output layer of our network.


In the cell below you should create the $Q$-Network and the target $\hat Q$-Network using the model architecture described above. Remember that both the $Q$-Network and the target $\hat Q$-Network have the same architecture.

Lastly, you should set `Adam` as the optimizer with a learning rate equal to `ALPHA`. Recall that `ALPHA` was defined in the [Hyperparameters](#2) section. We should note that for this exercise you should use the already imported packages:
```python
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
```

### 练习 1

在本练习中，你将创建 $Q$ 网络和目标 $\hat Q$ 网络，并设置优化器。请记住，深度 $Q$ 网络（DQN）是一个近似动作价值函数 $Q(s,a)\approx Q^*(s,a)$ 的神经网络。它通过学习如何将状态映射到 $Q$ 值来实现这一目标。

为了解决月球着陆器环境，我们将采用具有以下架构的 DQN：

* 一个接收 `state_size` 作为输入的 `Input` 层。

* 一个包含 `64` 个单元且激活函数为 `relu` 的 `Dense` 层。

* 一个包含 `64` 个单元且激活函数为 `relu` 的 `Dense` 层。

* 一个包含 `num_actions` 个单元且激活函数为 `linear` 的 `Dense` 层。这将是我们网络的输出层。


在下方单元格中，请使用上述模型架构创建 $Q$ 网络和目标 $\hat Q$ 网络。请记住，$Q$ 网络和目标 $\hat Q$ 网络具有相同的架构。

最后，请将 `Adam` 设置为优化器，并将学习率设为 `ALPHA`。回想一下，`ALPHA` 已在 [超参数](#2) 一节中定义。请注意，本练习应使用已经导入的软件包：
```python
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
```

In [ ]:
# UNQ_C1
# GRADED CELL

# Create the Q-Network
q_network = Sequential([
    ### START CODE HERE ### 

    ### END CODE HERE ### 
    ])

# Create the target Q^-Network
target_q_network = Sequential([
    ### START CODE HERE ### 

    ### END CODE HERE ###
    ])

### START CODE HERE ### 
optimizer = None
### END CODE HERE ###

In [ ]:
# UNIT TEST
from public_tests import *

test_network(q_network)
test_network(target_q_network)
test_optimizer(optimizer, ALPHA) 

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
```python
# Create the Q-Network
q_network = Sequential([
    Input(shape=state_size),                      
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'),
    ])

# Create the target Q^-Network
target_q_network = Sequential([
    Input(shape=state_size),                       
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'), 
    ])

optimizer = Adam(learning_rate=ALPHA)                                  
``` 

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
```python
# Create the Q-Network
q_network = Sequential([
    Input(shape=state_size),                      
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'),
    ])

# Create the target Q^-Network
target_q_network = Sequential([
    Input(shape=state_size),                       
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'), 
    ])

optimizer = Adam(learning_rate=ALPHA)                                  
```
</details>

<a name="6.2"></a>
### 6.2 Experience Replay

When an agent interacts with the environment, the states, actions, and rewards the agent experiences are sequential by nature. If the agent tries to learn from these consecutive experiences it can run into problems due to the strong correlations between them. To avoid this, we employ a technique known as **Experience Replay** to generate uncorrelated experiences for training our agent. Experience replay consists of storing the agent's experiences (i.e the states, actions, and rewards the agent receives) in a memory buffer and then sampling a random mini-batch of experiences from the buffer to do the learning. The experience tuples $(S_t, A_t, R_t, S_{t+1})$ will be added to the memory buffer at each time step as the agent interacts with the environment.

For convenience, we will store the experiences as named tuples.

### 6.2 经验回放（Experience Replay）

当智能体与环境交互时，它所经历的状态、动作和奖励本质上是按时间顺序连续的。如果智能体尝试直接从这些连续的经验中学习，由于它们之间存在很强的相关性，可能会遇到问题。为了避免这种情况，我们采用一种称为**经验回放（Experience Replay）**的技术，以生成不相关的经验来训练智能体。经验回放的做法是将智能体的经验（即智能体收到的状态、动作和奖励）存储在记忆缓冲区（memory buffer）中，然后从缓冲区中随机采样一个小批量（mini-batch）的经验进行学习。在智能体与环境交互的每个时间步，经验元组 $(S_t, A_t, R_t, S_{t+1})$ 都会被添加到记忆缓冲区中。

为了方便起见，我们将经验存储为具名元组（named tuples）。

In [ ]:
# Store experiences as named tuples
experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])

By using experience replay we avoid problematic correlations, oscillations and instabilities. In addition, experience replay also allows the agent to potentially use the same experience in multiple weight updates, which increases data efficiency.

通过使用经验回放，我们可以避免有害的相关性、震荡和不稳定性。此外，经验回放还允许智能体在多次权重更新中重复利用相同的经验，从而提高了数据效率。

<a name="7"></a>
## 7 - Deep Q-Learning Algorithm with Experience Replay

Now that we know all the techniques that we are going to use, we can put them togther to arrive at the Deep Q-Learning Algorithm With Experience Replay.
<br>
<br>
<figure>
  <img src = "images/deep_q_algorithm.png" width = 90% style = "border: thin silver solid; padding: 0px">
      <figcaption style = "text-align: center; font-style: italic">Fig 3. Deep Q-Learning with Experience Replay.</figcaption>
</figure>

## 7 - 带经验回放的深度 Q 学习算法

既然我们已经了解了将要使用的所有技术，就可以将它们组合起来，得到带经验回放的深度 Q 学习算法（Deep Q-Learning Algorithm With Experience Replay）。

<a name="ex02"></a>
### Exercise 2

In this exercise you will implement line ***12*** of the algorithm outlined in *Fig 3* above and you will also compute the loss between the $y$ targets and the $Q(s,a)$ values. In the cell below, complete the `compute_loss` function by setting the $y$ targets equal to:

$$
\begin{equation}
    y_j =
    \begin{cases}
      R_j & \text{if episode terminates at step  } j+1\\
      R_j + \gamma \max_{a'}\hat{Q}(s_{j+1},a') & \text{otherwise}\\
    \end{cases}       
\end{equation}
$$

Here are a couple of things to note:

* The `compute_loss` function takes in a mini-batch of experience tuples. This mini-batch of experience tuples is unpacked to extract the `states`, `actions`, `rewards`, `next_states`, and `done_vals`. You should keep in mind that these variables are *TensorFlow Tensors* whose size will depend on the mini-batch size. For example, if the mini-batch size is `64` then both `rewards` and `done_vals` will be TensorFlow Tensors with `64` elements.


* Using `if/else` statements to set the $y$ targets will not work when the variables are tensors with many elements. However, notice that you can use the `done_vals` to implement the above in a single line of code. To do this, recall that the `done` variable is a Boolean variable that takes the value `True` when an episode terminates at step $j+1$ and it is `False` otherwise. Taking into account that a Boolean value of `True` has the numerical value of `1` and a Boolean value of `False` has the numerical value of `0`, you can use the factor `(1 - done_vals)` to implement the above in a single line of code. Here's a hint: notice that `(1 - done_vals)` has a value of `0` when `done_vals` is `True` and a value of `1` when `done_vals` is `False`. 

Lastly, compute the loss by calculating the Mean-Squared Error (`MSE`) between the `y_targets` and the `q_values`. To calculate the mean-squared error you should use the already imported package `MSE`:
```python
from tensorflow.keras.losses import MSE
```

### 练习 2

在本练习中，你将实现上面*图 3*所示算法的第 ***12*** 行，并计算目标值 $y$ 与 $Q(s,a)$ 值之间的损失。在下方单元格中，请通过按如下方式设置目标值 $y$ 来补全 `compute_loss` 函数：

$$
\begin{equation}
    y_j =
    \begin{cases}
      R_j & \text{if episode terminates at step  } j+1\\
      R_j + \gamma \max_{a'}\hat{Q}(s_{j+1},a') & \text{otherwise}\\
    \end{cases}       
\end{equation}
$$

以下是需要注意的几点：

* `compute_loss` 函数接收一个小批量的经验元组。这个小批量的经验元组会被解包以提取出 `states`、`actions`、`rewards`、`next_states` 和 `done_vals`。请记住，这些变量都是 *TensorFlow 张量（Tensors）*，其大小取决于小批量的大小。例如，如果小批量大小为 `64`，那么 `rewards` 和 `done_vals` 都将是包含 `64` 个元素的 TensorFlow 张量。


* 当变量是包含多个元素的张量时，使用 `if/else` 语句来设置目标值 $y$ 是行不通的。但请注意，你可以利用 `done_vals` 用一行代码实现上述逻辑。为此，请回想 `done` 变量是一个布尔变量：当回合在第 $j+1$ 步终止时其值为 `True`，否则为 `False`。考虑到布尔值 `True` 的数值为 `1`，`False` 的数值为 `0`，你可以使用因子 `(1 - done_vals)` 在一行代码中实现上述公式。提示：注意当 `done_vals` 为 `True` 时 `(1 - done_vals)` 的值为 `0`，当 `done_vals` 为 `False` 时其值为 `1`。

最后，通过计算 `y_targets` 和 `q_values` 之间的均方误差（Mean-Squared Error，`MSE`）来求得损失。要计算均方误差，请使用已经导入的 `MSE`：
```python
from tensorflow.keras.losses import MSE
```

In [ ]:
# UNQ_C2
# GRADED FUNCTION: calculate_loss

def compute_loss(experiences, gamma, q_network, target_q_network):
    """ 
    Calculates the loss.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Karas model for predicting the targets
          
    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """
    
    # Unpack the mini-batch of experience tuples
    states, actions, rewards, next_states, done_vals = experiences
    
    # Compute max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # Set y = R if episode terminates, otherwise set y = R + γ max Q^(s,a).
    ### START CODE HERE ### 
    y_targets = None
    ### END CODE HERE ###
    
    # Get the q_values
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))
        
    # Compute the loss
    ### START CODE HERE ### 
    loss = None 
    ### END CODE HERE ### 
    
    return loss

In [ ]:
# UNIT TEST    
test_compute_loss(compute_loss)

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
```python
def compute_loss(experiences, gamma, q_network, target_q_network):
    """ 
    Calculates the loss.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Karas model for predicting the targets
          
    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """

    
    # Unpack the mini-batch of experience tuples
    states, actions, rewards, next_states, done_vals = experiences
    
    # Compute max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # Set y = R if episode terminates, otherwise set y = R + γ max Q^(s,a).
    y_targets = rewards + (gamma * max_qsa * (1 - done_vals))
    
    # Get the q_values
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))
    
    # Calculate the loss
    loss = MSE(y_targets, q_values)
    
    return loss

``` 
    

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
```python
def compute_loss(experiences, gamma, q_network, target_q_network):
    """ 
    Calculates the loss.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Karas model for predicting the targets
          
    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """

    
    # Unpack the mini-batch of experience tuples
    states, actions, rewards, next_states, done_vals = experiences
    
    # Compute max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # Set y = R if episode terminates, otherwise set y = R + γ max Q^(s,a).
    y_targets = rewards + (gamma * max_qsa * (1 - done_vals))
    
    # Get the q_values
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))
    
    # Calculate the loss
    loss = MSE(y_targets, q_values)
    
    return loss

```
</details>

<a name="8"></a>
## 8 - Update the Network Weights

We will use the `agent_learn` function below to implement lines ***12 -14*** of the algorithm outlined in [Fig 3](#7). The `agent_learn` function will update the weights of the $Q$ and target $\hat Q$ networks using a custom training loop. Because we are using a custom training loop we need to retrieve the gradients via a `tf.GradientTape` instance, and then call `optimizer.apply_gradients()` to update the weights of our $Q$-Network. Note that we are also using the `@tf.function` decorator to increase performance. Without this decorator our training will take twice as long. If you would like to know more about how to increase performance with `@tf.function` take a look at the [TensorFlow documentation](https://www.tensorflow.org/guide/function).

The last line of this function updates the weights of the target $\hat Q$-Network using a [soft update](#6.1). If you want to know how this is implemented in code we encourage you to take a look at the `utils.update_target_network` function in the `utils` module.

## 8 - 更新网络权重

我们将使用下面的 `agent_learn` 函数来实现 [图 3](#7) 所示算法的第 ***12 - 14*** 行。`agent_learn` 函数将使用自定义训练循环来更新 $Q$ 网络和目标 $\hat Q$ 网络的权重。由于我们使用的是自定义训练循环，因此需要通过 `tf.GradientTape` 实例获取梯度，然后调用 `optimizer.apply_gradients()` 来更新 $Q$ 网络的权重。请注意，我们还使用了 `@tf.function` 装饰器来提高性能。如果没有这个装饰器，训练耗时将会翻倍。如果你想了解更多关于如何使用 `@tf.function` 提升性能的内容，可以查看 [TensorFlow 文档](https://www.tensorflow.org/guide/function)。

该函数的最后一行使用 [软更新](#6.1) 来更新目标 $\hat Q$ 网络的权重。如果你想了解这在代码中是如何实现的，我们建议查看 `utils` 模块中的 `utils.update_target_network` 函数。

In [ ]:
@tf.function
def agent_learn(experiences, gamma):
    """
    Updates the weights of the Q networks.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
    
    """
    
    # Calculate the loss
    with tf.GradientTape() as tape:
        loss = compute_loss(experiences, gamma, q_network, target_q_network)

    # Get the gradients of the loss with respect to the weights.
    gradients = tape.gradient(loss, q_network.trainable_variables)
    
    # Update the weights of the q_network.
    optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))

    # update the weights of target q_network
    utils.update_target_network(q_network, target_q_network)

<a name="9"></a>
## 9 - Train the Agent

We are now ready to train our agent to solve the Lunar Lander environment. In the cell below we will implement the algorithm in [Fig 3](#7) line by line (please note that we have included the same algorithm below for easy reference. This will prevent you from scrolling up and down the notebook):

* **Line 1**: We initialize the `memory_buffer` with a capacity of $N =$ `MEMORY_SIZE`. Notice that we are using a `deque` as the data structure for our `memory_buffer`.


* **Line 2**: We skip this line since we already initialized the `q_network` in [Exercise 1](#ex01).


* **Line 3**: We initialize the `target_q_network` by setting its weights to be equal to those of the `q_network`.


* **Line 4**: We start the outer loop. Notice that we have set $M =$ `num_episodes = 2000`. This number is reasonable because the agent should be able to solve the Lunar Lander environment in less than `2000` episodes using this notebook's default parameters.


* **Line 5**: We use the `.reset()` method to reset the environment to the initial state and get the initial state.


* **Line 6**: We start the inner loop. Notice that we have set $T =$ `max_num_timesteps = 1000`. This means that the episode will automatically terminate if the episode hasn't terminated after `1000` time steps.


* **Line 7**: The agent observes the current `state` and chooses an `action` using an $\epsilon$-greedy policy. Our agent starts out using a value of $\epsilon =$ `epsilon = 1` which yields an $\epsilon$-greedy policy that is equivalent to the equiprobable random policy. This means that at the beginning of our training, the agent is just going to take random actions regardless of the observed `state`. As training progresses we will decrease the value of $\epsilon$ slowly towards a minimum value using a given $\epsilon$-decay rate. We want this minimum value to be close to zero because a value of $\epsilon = 0$ will yield an $\epsilon$-greedy policy that is equivalent to the greedy policy. This means that towards the end of training, the agent will lean towards selecting the `action` that it believes (based on its past experiences) will maximize $Q(s,a)$. We will set the minimum $\epsilon$ value to be `0.01` and not exactly 0 because we always want to keep a little bit of exploration during training. If you want to know how this is implemented in code we encourage you to take a look at the `utils.get_action` function in the `utils` module.


* **Line 8**: We use the `.step()` method to take the given `action` in the environment and get the `reward` and the `next_state`. 


* **Line 9**: We store the `experience(state, action, reward, next_state, done)` tuple in our `memory_buffer`. Notice that we also store the `done` variable so that we can keep track of when an episode terminates. This allowed us to set the $y$ targets in [Exercise 2](#ex02).


* **Line 10**: We check if the conditions are met to perform a learning update. We do this by using our custom `utils.check_update_conditions` function. This function checks if $C =$ `NUM_STEPS_FOR_UPDATE = 4` time steps have occured and if our `memory_buffer` has enough experience tuples to fill a mini-batch. For example, if the mini-batch size is `64`, then our `memory_buffer` should have at least `64` experience tuples in order to pass the latter condition. If the conditions are met, then the `utils.check_update_conditions` function will return a value of `True`, otherwise it will return a value of `False`.


* **Lines 11 - 14**: If the `update` variable is `True` then we perform a learning update. The learning update consists of sampling a random mini-batch of experience tuples from our `memory_buffer`, setting the $y$ targets, performing gradient descent, and updating the weights of the networks. We will use the `agent_learn` function we defined in [Section 8](#8) to perform the latter 3.


* **Line 15**: At the end of each iteration of the inner loop we set `next_state` as our new `state` so that the loop can start again from this new state. In addition, we check if the episode has reached a terminal state (i.e we check if `done = True`). If a terminal state has been reached, then we break out of the inner loop.


* **Line 16**: At the end of each iteration of the outer loop we update the value of $\epsilon$, and check if the environment has been solved. We consider that the environment has been solved if the agent receives an average of `200` points in the last `100` episodes. If the environment has not been solved we continue the outer loop and start a new episode.

Finally, we wanted to note that we have included some extra variables to keep track of the total number of points the agent received in each episode. This will help us determine if the agent has solved the environment and it will also allow us to see how our agent performed during training. We also use the `time` module to measure how long the training takes. 

<br>
<br>
<figure>
  <img src = "images/deep_q_algorithm.png" width = 90% style = "border: thin silver solid; padding: 0px">
      <figcaption style = "text-align: center; font-style: italic">Fig 4. Deep Q-Learning with Experience Replay.</figcaption>
</figure>
<br>

**Note:** With this notebook's default parameters, the following cell takes between 10 to 15 minutes to run. 

## 9 - 训练智能体

现在我们已经准备好训练智能体来解决月球着陆器环境了。在下方单元格中，我们将逐行实现 [图 3](#7) 中的算法（请注意，为了方便查阅、避免你在 notebook 中反复上下滚动，我们在下方再次附上了相同的算法图）：

* **第 1 行**：我们初始化容量为 $N =$ `MEMORY_SIZE` 的 `memory_buffer`。注意我们使用 `deque` 作为 `memory_buffer` 的数据结构。


* **第 2 行**：由于我们已经在 [练习 1](#ex01) 中初始化了 `q_network`，因此跳过这一行。


* **第 3 行**：通过将 `target_q_network` 的权重设为与 `q_network` 相同来对其进行初始化。


* **第 4 行**：开始外层循环。注意我们设置了 $M =$ `num_episodes = 2000`。这个数字是合理的，因为使用本 notebook 的默认参数，智能体应该能够在不到 `2000` 个回合内解决月球着陆器环境。


* **第 5 行**：使用 `.reset()` 方法将环境重置到初始状态并获取初始状态。


* **第 6 行**：开始内层循环。注意我们设置了 $T =$ `max_num_timesteps = 1000`。这意味着如果一个回合在 `1000` 个时间步后仍未终止，它将自动终止。


* **第 7 行**：智能体观测当前 `state`，并使用 $\epsilon$-贪婪策略（$\epsilon$-greedy policy）选择一个 `action`。我们的智能体最初使用 $\epsilon =$ `epsilon = 1`，此时 $\epsilon$-贪婪策略等价于等概率随机策略。这意味着在训练初期，无论观测到的 `state` 是什么，智能体都会采取随机动作。随着训练的进行，我们将使用给定的 $\epsilon$-衰减率将 $\epsilon$ 的值缓慢降低到一个最小值。我们希望这个最小值接近于零，因为 $\epsilon = 0$ 会使 $\epsilon$-贪婪策略等价于贪婪策略。这意味着在训练后期，智能体将倾向于选择它认为（基于过去经验）能够最大化 $Q(s,a)$ 的 `action`。我们将最小 $\epsilon$ 值设为 `0.01` 而不是精确的 0，因为我们始终希望在训练过程中保留一点探索。如果你想了解这在代码中是如何实现的，建议查看 `utils` 模块中的 `utils.get_action` 函数。


* **第 8 行**：使用 `.step()` 方法在环境中执行给定的 `action`，并获取 `reward` 和 `next_state`。


* **第 9 行**：将 `experience(state, action, reward, next_state, done)` 元组存入 `memory_buffer`。注意我们也存储了 `done` 变量，以便跟踪回合何时终止。这使我们能够在 [练习 2](#ex02) 中正确设置目标值 $y$。


* **第 10 行**：检查是否满足执行学习更新的条件。我们通过自定义的 `utils.check_update_conditions` 函数来实现这一点。该函数检查是否已经经过了 $C =$ `NUM_STEPS_FOR_UPDATE = 4` 个时间步，以及 `memory_buffer` 中是否有足够多的经验元组来组成一个小批量。例如，如果小批量大小为 `64`，那么 `memory_buffer` 中至少需要有 `64` 个经验元组才能满足后一个条件。如果条件均满足，`utils.check_update_conditions` 函数将返回 `True`，否则返回 `False`。


* **第 11 - 14 行**：如果 `update` 变量为 `True`，则执行一次学习更新。学习更新包括从 `memory_buffer` 中随机采样一个小批量的经验元组、设置目标值 $y$、执行梯度下降以及更新网络权重。我们将使用在 [第 8 节](#8) 中定义的 `agent_learn` 函数来执行后三步。


* **第 15 行**：在内层循环每次迭代结束时，我们将 `next_state` 设为新的 `state`，以便循环从这个新状态继续。此外，我们检查回合是否达到了终止状态（即检查 `done = True`）。如果达到了终止状态，则跳出内层循环。


* **第 16 行**：在外层循环每次迭代结束时，我们更新 $\epsilon$ 的值，并检查环境是否已被解决。如果智能体在最近 `100` 个回合中获得的平均分达到 `200` 分，我们就认为环境已被解决。如果环境尚未解决，则继续外层循环并开始一个新的回合。

最后，需要说明的是，我们还加入了一些额外的变量来记录智能体在每个回合中获得的总分。这将帮助我们判断智能体是否解决了环境，并让我们能够观察智能体在训练过程中的表现。我们还使用 `time` 模块来测量训练所需的时长。

**注意：** 使用本 notebook 的默认参数，下方单元格需要运行 10 到 15 分钟。

In [ ]:
start = time.time()

num_episodes = 2000
max_num_timesteps = 1000

total_point_history = []

num_p_av = 100    # number of total points to use for averaging
epsilon = 1.0     # initial ε value for ε-greedy policy

# Create a memory buffer D with capacity N
memory_buffer = deque(maxlen=MEMORY_SIZE)

# Set the target network weights equal to the Q-Network weights
target_q_network.set_weights(q_network.get_weights())

for i in range(num_episodes):
    
    # Reset the environment to the initial state and get the initial state
    state = env.reset()
    total_points = 0
    
    for t in range(max_num_timesteps):
        
        # From the current state S choose an action A using an ε-greedy policy
        state_qn = np.expand_dims(state, axis=0)  # state needs to be the right shape for the q_network
        q_values = q_network(state_qn)
        action = utils.get_action(q_values, epsilon)
        
        # Take action A and receive reward R and the next state S'
        next_state, reward, done, _ = env.step(action)
        
        # Store experience tuple (S,A,R,S') in the memory buffer.
        # We store the done variable as well for convenience.
        memory_buffer.append(experience(state, action, reward, next_state, done))
        
        # Only update the network every NUM_STEPS_FOR_UPDATE time steps.
        update = utils.check_update_conditions(t, NUM_STEPS_FOR_UPDATE, memory_buffer)
        
        if update:
            # Sample random mini-batch of experience tuples (S,A,R,S') from D
            experiences = utils.get_experiences(memory_buffer)
            
            # Set the y targets, perform a gradient descent step,
            # and update the network weights.
            agent_learn(experiences, GAMMA)
        
        state = next_state.copy()
        total_points += reward
        
        if done:
            break
            
    total_point_history.append(total_points)
    av_latest_points = np.mean(total_point_history[-num_p_av:])
    
    # Update the ε value
    epsilon = utils.get_new_eps(epsilon)

    print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}", end="")

    if (i+1) % num_p_av == 0:
        print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}")

    # We will consider that the environment is solved if we get an
    # average of 200 points in the last 100 episodes.
    if av_latest_points >= 200.0:
        print(f"\n\nEnvironment solved in {i+1} episodes!")
        q_network.save('lunar_lander_model.h5')
        break
        
tot_time = time.time() - start

print(f"\nTotal Runtime: {tot_time:.2f} s ({(tot_time/60):.2f} min)")

We can plot the point history to see how our agent improved during training.

我们可以绘制得分历史曲线，查看智能体在训练过程中的进步情况。

In [ ]:
# Plot the point history
utils.plot_history(total_point_history)

<a name="10"></a>
## 10 - See the Trained Agent In Action

Now that we have trained our agent, we can see it in action. We will use the `utils.create_video` function to create a video of our agent interacting with the environment using the trained $Q$-Network. The `utils.create_video` function uses the `imageio` library to create the video. This library produces some warnings that can be distracting, so, to suppress these warnings we run the code below.

## 10 - 查看训练好的智能体的实际表现

既然已经训练好了智能体，现在就可以看看它的实际表现了。我们将使用 `utils.create_video` 函数，利用训练好的 $Q$ 网络生成一段智能体与环境交互的视频。`utils.create_video` 函数使用 `imageio` 库来创建视频。该库会产生一些可能干扰视线的警告信息，因此我们运行下方代码来屏蔽这些警告。

In [ ]:
# Suppress warnings from imageio
import logging
logging.getLogger().setLevel(logging.ERROR)

In the cell below we create a video of our agent interacting with the Lunar Lander environment using the trained `q_network`. The video is saved to the `videos` folder with the given `filename`. We use the `utils.embed_mp4` function to embed the video in the Jupyter Notebook so that we can see it here directly without having to download it.

We should note that since the lunar lander starts with a random initial force applied to its center of mass, every time you run the cell below you will see a different video. If the agent was trained properly, it should be able to land the lunar lander in the landing pad every time, regardless of the initial force applied to its center of mass.

在下方单元格中，我们使用训练好的 `q_network` 创建一段智能体与月球着陆器环境交互的视频。视频将以给定的 `filename` 保存在 `videos` 文件夹中。我们使用 `utils.embed_mp4` 函数将视频嵌入到 Jupyter Notebook 中，这样就可以直接在这里观看而无需下载。

请注意，由于月球着陆器在初始时其质心会受到随机的初始作用力，因此每次运行下方单元格时你都会看到不同的视频。如果智能体训练得当，无论质心受到怎样的初始作用力，它每次都应该能够将月球着陆器降落在着陆坪上。

In [ ]:
filename = "./videos/lunar_lander.mp4"

utils.create_video(filename, env, q_network)
utils.embed_mp4(filename)

<a name="11"></a>
## 11 - Congratulations!

You have successfully used Deep Q-Learning with Experience Replay to train an agent to land a lunar lander safely on a landing pad on the surface of the moon. Congratulations!

## 11 - 恭喜！

你已经成功使用带经验回放的深度 Q 学习训练了一个智能体，将月球着陆器安全地降落在月球表面的着陆坪上。恭喜你！

<a name="12"></a>
## 12 - References

If you would like to learn more about Deep Q-Learning, we recommend you check out the following papers.


* [Human-level Control Through Deep Reinforcement Learning](https://storage.googleapis.com/deepmind-media/dqn/DQNNaturePaper.pdf)


* [Continuous Control with Deep Reinforcement Learning](https://arxiv.org/pdf/1509.02971.pdf)


* [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf)

## 12 - 参考文献

如果你想进一步了解深度 Q 学习，我们推荐阅读以下论文。


* [Human-level Control Through Deep Reinforcement Learning](https://storage.googleapis.com/deepmind-media/dqn/DQNNaturePaper.pdf)


* [Continuous Control with Deep Reinforcement Learning](https://arxiv.org/pdf/1509.02971.pdf)


* [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf)